# Multi-Frame CRNN Training Walkthrough

This notebook breaks down the training pipeline for a Multi-Frame CRNN model used for text recognition from video tracks. We will go through each component step-by-step.

## 1. Import library

In [12]:
import os
import glob
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import swin_t, Swin_T_Weights
from PIL import Image
import numpy as np
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import math
import torch.backends.cudnn as cudnn
import albumentations as A
from albumentations.pytorch import ToTensorV2

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🔒 Đã cố định Seed: {seed}")

cudnn.benchmark = True

## 2. Configuration

In [ ]:
class Config:
    DATA_ROOT = "../data/train"
    IMG_HEIGHT = 32
    IMG_WIDTH = 128
    CHARS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-"
    BATCH_SIZE = 128
    LEARNING_RATE = 0.005
    EPOCHS = 50
    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    CHAR2IDX = {char: idx + 1 for idx, char in enumerate(CHARS)}
    IDX2CHAR = {idx + 1: char for idx, char in enumerate(CHARS)}
    NUM_CLASSES = len(CHARS) + 1
    VAL_SPLIT_FILE = "val_tracks.json"
    TEST_SPLIT_FILE = "test_tracks.json"
    VAL_SIZE = 2000
    TEST_SIZE = 2000

## 3. Data Augmentation
Define transformations for training (with augmentation) and validation/testing.

In [14]:
def get_train_transforms():
    return A.Compose([
        A.Resize(height=Config.IMG_HEIGHT, width=Config.IMG_WIDTH),
        A.OneOf([
            A.Affine(scale=(0.8, 1.2), translate_percent=(0.1, 0.1), rotate=(-15, 15), shear=(-10, 10), p=1.0, fill=128),
            A.ElasticTransform(alpha=1, sigma=50, p=1.0),
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
            A.OpticalDistortion(distort_limit=0.3, p=1.0),
        ], p=0.7),
        A.Perspective(scale=(0.05, 0.15), p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1.0),
        ], p=0.6),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.GaussNoise(std_range=(0.1, 0.5), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
        ], p=0.5),
        A.CoarseDropout(
            num_holes_range=(2, 8),
            hole_height_range=(4, 8),
            hole_width_range=(4, 8),
            fill=128,
            p=0.5
        ),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])

def get_degradation_transforms():
    return A.Compose([
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.Defocus(radius=(1, 3), alias_blur=(0.1, 0.3), p=1.0),
        ], p=0.8),
        A.OneOf([
            A.GaussNoise(std_range=(0.1, 0.5), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
        ], p=0.8),
        A.ImageCompression(quality_range=(10, 50), p=0.5),
        A.Downscale(scale_range=(0.25, 0.5), p=0.5),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(height=Config.IMG_HEIGHT, width=Config.IMG_WIDTH),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])

## 4. Dataset Class
Custom Dataset class to handle multi-frame loading and splitting.

In [15]:
class AdvancedMultiFrameDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.mode = mode
        self.samples = []

        if mode == 'train':
            self.transform = get_train_transforms()
            self.degrade = get_degradation_transforms()
        else:
            self.transform = get_val_transforms()
            self.degrade = None

        print(f"[{mode.upper()}] Scanning: {root_dir}")
        abs_root = os.path.abspath(root_dir)
        search_path = os.path.join(abs_root, "**", "track_*")
        all_tracks = sorted(glob.glob(search_path, recursive=True))

        if not all_tracks:
            print("❌ LỖI: Không tìm thấy data.")
            return

        train_tracks = []
        val_tracks = []
        test_tracks = []

        # Check if split files exist
        val_exists = os.path.exists(Config.VAL_SPLIT_FILE)
        test_exists = os.path.exists(Config.TEST_SPLIT_FILE)

        val_ids = set()
        test_ids = set()

        if val_exists and test_exists:
            print(f"📂 Loading splits from '{Config.VAL_SPLIT_FILE}' and '{Config.TEST_SPLIT_FILE}'...")
            try:
                with open(Config.VAL_SPLIT_FILE, 'r') as f:
                    val_ids = set(json.load(f))
                with open(Config.TEST_SPLIT_FILE, 'r') as f:
                    test_ids = set(json.load(f))
            except:
                val_ids = set()
                test_ids = set()
                print("⚠️ Lỗi đọc file split, sẽ tạo lại.")

            for t in all_tracks:
                track_name = os.path.basename(t)
                if track_name in val_ids:
                    val_tracks.append(t)
                elif track_name in test_ids:
                    test_tracks.append(t)
                else:
                    train_tracks.append(t)

            # Nếu split không khớp, tạo lại
            if (not val_tracks or not test_tracks) and len(all_tracks) > 0:
                print("⚠️ File split không khớp với dữ liệu hiện tại. Chia lại...")
                val_ids = set()
                test_ids = set()

        if not val_ids or not test_ids:
            print(f"⚠️ Creating new split: {Config.VAL_SIZE} val, {Config.TEST_SIZE} test...")
            random.Random(Config.SEED).shuffle(all_tracks)

            # Chia: val 2000, test 2000, train còn lại
            val_tracks = all_tracks[:Config.VAL_SIZE]
            test_tracks = all_tracks[Config.VAL_SIZE:Config.VAL_SIZE + Config.TEST_SIZE]
            train_tracks = all_tracks[Config.VAL_SIZE + Config.TEST_SIZE:]

            # Lưu split files
            val_ids = [os.path.basename(t) for t in val_tracks]
            test_ids = [os.path.basename(t) for t in test_tracks]
            with open(Config.VAL_SPLIT_FILE, 'w') as f:
                json.dump(val_ids, f, indent=2)
            with open(Config.TEST_SPLIT_FILE, 'w') as f:
                json.dump(test_ids, f, indent=2)
            print(f"✅ Saved splits: val={len(val_tracks)}, test={len(test_tracks)}, train={len(train_tracks)}")

        if mode == 'train':
            selected_tracks = train_tracks
        elif mode == 'val':
            selected_tracks = val_tracks
        else:  # mode == 'test'
            selected_tracks = test_tracks
        print(f"[{mode.upper()}] Loaded {len(selected_tracks)} tracks.")

        for track_path in tqdm(selected_tracks, desc=f"Indexing {mode}"):
            json_path = os.path.join(track_path, "annotations.json")
            if not os.path.exists(json_path): continue
            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                if isinstance(data, list): data = data[0]
                label = data.get('plate_text', data.get('license_plate', data.get('text', '')))
                if not label: continue

                lr_files = sorted(glob.glob(os.path.join(track_path, "lr-*.png")) + glob.glob(os.path.join(track_path, "lr-*.jpg")))
                hr_files = sorted(glob.glob(os.path.join(track_path, "hr-*.png")) + glob.glob(os.path.join(track_path, "hr-*.jpg")))

                if len(lr_files) > 0:
                    self.samples.append({
                        'lr_paths': lr_files,
                        'hr_paths': hr_files,
                        'label': label
                    })
            except: pass

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        label = item['label']

        use_hr = (self.mode == 'train') and (len(item['hr_paths']) > 0) and (random.random() < 0.5)

        if use_hr:
            img_paths = item['hr_paths']
            if len(img_paths) < 5: img_paths = img_paths + [img_paths[-1]] * (5 - len(img_paths))
            else: img_paths = img_paths[:5]

            images_list = []
            for p in img_paths:
                image = cv2.imread(p)
                if image is None: image = np.zeros((Config.IMG_HEIGHT, Config.IMG_WIDTH, 3), dtype=np.uint8)
                else: image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                if self.degrade:
                    image = self.degrade(image=image)['image']
                image = self.transform(image=image)['image']
                images_list.append(image)
        else:
            img_paths = item['lr_paths']
            if len(img_paths) < 5: img_paths = img_paths + [img_paths[-1]] * (5 - len(img_paths))
            else: img_paths = img_paths[:5]

            images_list = []
            for p in img_paths:
                image = cv2.imread(p)
                if image is None: image = np.zeros((Config.IMG_HEIGHT, Config.IMG_WIDTH, 3), dtype=np.uint8)
                else: image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                image = self.transform(image=image)['image']
                images_list.append(image)

        images_tensor = torch.stack(images_list, dim=0)
        target = [Config.CHAR2IDX[c] for c in label if c in Config.CHAR2IDX]
        if len(target) == 0: target = [0]

        return images_tensor, torch.tensor(target, dtype=torch.long), len(target), label

    @staticmethod
    def collate_fn(batch):
        images, targets, target_lengths, labels_text = zip(*batch)
        images = torch.stack(images, 0)
        targets = torch.cat(targets)
        target_lengths = torch.tensor(target_lengths, dtype=torch.long)
        return images, targets, target_lengths, labels_text

## 5. Model Architecture
Define the Attention Fusion module and the main CRNN model.

In [16]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: [Batch, Seq_Len, d_model]
        return x + self.pe[:, :x.size(1)]

class ChannelSpatialFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.spatial_att = nn.Sequential(
            nn.Conv2d(channels, channels // 2, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(channels // 2, 1, 3, 1, 1)
        )
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.channel_att = nn.Sequential(
            nn.Linear(channels, channels // 4),
            nn.ReLU(),
            nn.Linear(channels // 4, channels),
            nn.Sigmoid()
        )
        self.fusion_conv = nn.Conv2d(channels, channels, 1)

    # SỬA: Thêm tham số t (số lượng frames) vào forward để không bị cứng t=5
    def forward(self, x, t):
        bt, c, h, w = x.size()
        b = bt // t

        # Reshape về dạng [Batch, Time, Channel, H, W] để tính attention giữa các frames
        x_view = x.view(b, t, c, h, w)

        # Spatial Attention
        scores = self.spatial_att(x).view(b, t, 1, h, w)
        att_map = F.softmax(scores, dim=1) # Softmax trên chiều thời gian (frames)

        # Fuse: Cộng gộp các frame lại dựa trên attention map
        fused_spatial = torch.sum(x_view * att_map, dim=1) # Kết quả: [B, C, H, W]

        # Channel Attention
        y = self.avg_pool(fused_spatial).view(b, c)
        y = self.channel_att(y).view(b, c, 1, 1) # [B, C, 1, 1]

        # Kết hợp Spatial và Channel
        return self.fusion_conv(fused_spatial * y)

class MultiFrameCRNN(nn.Module):
    def __init__(self, num_classes, d_model=512):
        super().__init__()
        # Backbone MobileNetV3
        mobilenet = swin_t(weights=Swin_T_Weights.DEFAULT)
        self.backbone = mobilenet.features # Output channels: 960. Downsample: 32x

        # --- PHẦN SỬA LỖI Ở ĐÂY ---
        self.conv_proj = nn.Sequential(
            # 1. Upsample chiều ngang (Width) lên 4 lần.
            # Ví dụ: Feature width đang là 4 -> sẽ thành 16.
            # 16 time-steps là đủ lớn so với target length (7).
            nn.Upsample(scale_factor=(1, 4), mode='bilinear', align_corners=True),

            # 2. Conv giảm chiều cao (H/2), giữ nguyên chiều rộng mới (W/1)
            # Đổi kernel_size=(3,3) và padding=(1,1) để an toàn hơn
            nn.Conv2d(960, d_model, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)),
            nn.BatchNorm2d(d_model),
            nn.ReLU()
        )
        # ---------------------------

        self.fusion = ChannelSpatialFusion(channels=d_model)
        self.pos_encoder = PositionalEncoding(d_model=d_model)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=d_model*4, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        # Output Layer
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        # Input shape: [Batch, Frames, Channels, Height, Width]
        b, t, c, h, w = x.size()

        # Gộp Batch và Frames
        x = x.view(b * t, c, h, w)

        # 1. Feature Extraction
        features = self.backbone(x) # Shape: [B*T, 960, H/32, W/32] -> Width rất nhỏ (ví dụ 4)

        # Sau khi qua conv_proj đã sửa: Width sẽ nhân 4 -> Shape: [B*T, d_model, H/64, W*4/32]
        features = self.conv_proj(features)

        # 2. Fusion
        fused = self.fusion(features, t)

        # 3. Chuyển đổi sang Sequence
        fused = F.adaptive_avg_pool2d(fused, (1, None))

        # [Batch, Width_New, d_model]
        seq = fused.squeeze(2).permute(0, 2, 1)

        # 4. Transformer & Output
        seq = self.pos_encoder(seq)
        out = self.transformer(seq)
        out = self.fc(out)
        return out.log_softmax(2)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class HybridAttentionFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.temporal_pool = nn.AdaptiveAvgPool2d(1)
        self.temporal_gate = nn.Sequential(
            nn.Linear(channels, channels // 4),
            nn.ReLU(True),
            nn.Linear(channels // 4, 1),
            nn.Sigmoid()
        )
        self.spatial_net = nn.Sequential(
            nn.Conv2d(channels * 2, channels // 2, 3, 1, 1),
            nn.BatchNorm2d(channels // 2),
            nn.ReLU(True),
            nn.Conv2d(channels // 2, 1, 3, 1, 1)
        )
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 4, 1),
            nn.ReLU(True),
            nn.Conv2d(channels // 4, channels, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)

    def forward(self, x, t):
        bt, c, h, w = x.size()
        b = bt // t
        x_view = x.view(b, t, c, h, w)
        pooled = self.temporal_pool(x).view(bt, c)
        t_weights = self.temporal_gate(pooled).view(b, t, 1, 1, 1)
        ref_idx = t // 2
        ref = x_view[:, ref_idx:ref_idx+1].repeat(1, t, 1, 1, 1)
        concat_feat = torch.cat([x_view, ref], dim=2).view(bt, c*2, h, w)
        s_scores = self.spatial_net(concat_feat).view(b, t, 1, h, w)
        combined_scores = s_scores * t_weights
        att_map = F.softmax(combined_scores, dim=1)
        fused = torch.sum(x_view * att_map, dim=1)
        c_att = self.channel_gate(fused)
        return fused * c_att

class MultiFrameCRNN(nn.Module):
    def __init__(self, num_classes, d_model=512):
        super().__init__()
        backbone_model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        self.backbone = backbone_model.features
        self.conv_proj = nn.Sequential(
            nn.Upsample(scale_factor=(1, 4), mode='bilinear', align_corners=True),
            nn.Conv2d(768, d_model, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)),
            nn.BatchNorm2d(d_model),
            nn.ReLU()
        )
        self.fusion = HybridAttentionFusion(channels=d_model)
        self.pos_encoder = PositionalEncoding(d_model=d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=d_model*4, dropout=0.15, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)
        self.fc = nn.Linear(d_model, num_classes)

    def freeze_backbone(self, freeze=True):
        for param in self.backbone.parameters():
            param.requires_grad = not freeze
        print(f"❄️ ConvNeXt Backbone: {'Frozen' if freeze else 'Unfrozen'}")

    def forward(self, x):
        b, t, c, h, w = x.size()
        x = x.view(b * t, c, h, w)
        features = self.backbone(x)
        features = self.conv_proj(features)
        fused = self.fusion(features, t)
        fused = F.adaptive_avg_pool2d(fused, (1, None))
        seq = fused.squeeze(2).permute(0, 2, 1)
        seq = self.pos_encoder(seq)
        out = self.transformer(seq)
        out = self.fc(out)
        return out.log_softmax(2)


In [17]:
def decode_predictions(preds, idx2char):
    result_list = []
    for p in preds:
        pred_str = ""
        last_char = 0
        for char_idx in p:
            c = char_idx.item()
            if c != 0 and c != last_char: pred_str += idx2char[c]
            last_char = c
        result_list.append(pred_str)
    return result_list

## 7. Training Pipeline
The main training loop, including validation and testing.

In [ ]:
def train_pipeline():
    seed_everything(Config.SEED)
    train_ds = AdvancedMultiFrameDataset(Config.DATA_ROOT, mode='train')
    val_ds = AdvancedMultiFrameDataset(Config.DATA_ROOT, mode='val')
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, collate_fn=AdvancedMultiFrameDataset.collate_fn, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=AdvancedMultiFrameDataset.collate_fn, num_workers=4) if val_ds else None

    model = MultiFrameCRNN(num_classes=Config.NUM_CLASSES).to(Config.DEVICE)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=Config.LEARNING_RATE, steps_per_epoch=len(train_loader), epochs=Config.EPOCHS, pct_start=0.3, div_factor=25.0)
    scaler = GradScaler()
    best_acc = 0.0

    for epoch in range(Config.EPOCHS):
        if epoch < 5:
            model.freeze_backbone(True)
        else:
            should_freeze = np.random.random() < 0.3
            model.freeze_backbone(should_freeze)

        model.train()
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{Config.EPOCHS}")
        for images, targets, target_lengths, _ in pbar:
            images, targets, target_lengths = images.to(Config.DEVICE), targets.to(Config.DEVICE), target_lengths.to(Config.DEVICE)
            optimizer.zero_grad(set_to_none=True)
            
            lam = np.random.beta(1.0, 1.0) if random.random() < 0.3 else 1.0
            if lam < 1.0:
                idx = torch.randperm(images.size(0)).to(Config.DEVICE)
                images = lam * images + (1 - lam) * images[idx]
            
            with autocast('cuda'):
                preds = model(images)
                loss = criterion(preds.permute(1, 0, 2), targets, torch.full((images.size(0),), preds.size(1), dtype=torch.long), target_lengths)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            epoch_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})

        if val_loader:
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for images, targets, target_lengths, labels in val_loader:
                    preds = model(images.to(Config.DEVICE))
                    decoded = decode_predictions(preds, Config.IDX2CHAR)
                    for p, g in zip(decoded, labels):
                        if p == g: correct += 1
                        total += 1
            val_acc = (correct / total) * 100
            print(f"Result: Train Loss: {epoch_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model.state_dict(), "best_model.pth")


In [ ]:
def train_pipeline():
    seed_everything(Config.SEED)
    train_ds = AdvancedMultiFrameDataset(Config.DATA_ROOT, mode='train')
    val_ds = AdvancedMultiFrameDataset(Config.DATA_ROOT, mode='val')
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, collate_fn=AdvancedMultiFrameDataset.collate_fn, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=AdvancedMultiFrameDataset.collate_fn, num_workers=4) if val_ds else None

    model = MultiFrameCRNN(num_classes=Config.NUM_CLASSES).to(Config.DEVICE)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=Config.LEARNING_RATE, steps_per_epoch=len(train_loader), epochs=Config.EPOCHS, pct_start=0.3)
    scaler = GradScaler()
    best_acc = 0.0

    for epoch in range(Config.EPOCHS):
        # ALL FREEZE REMOVED
        model.train()
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{Config.EPOCHS}")
        for images, targets, target_lengths, _ in pbar:
            images, targets, target_lengths = images.to(Config.DEVICE), targets.to(Config.DEVICE), target_lengths.to(Config.DEVICE)
            optimizer.zero_grad(set_to_none=True)
            
            # Optional Mixup check
            lam = np.random.beta(1.0, 1.0) if random.random() < 0.3 else 1.0
            if lam < 1.0:
                idx = torch.randperm(images.size(0)).to(Config.DEVICE)
                images = lam * images + (1 - lam) * images[idx]
            
            with autocast('cuda'):
                preds = model(images)
                loss = criterion(preds.permute(1, 0, 2), targets, torch.full((images.size(0),), preds.size(1), dtype=torch.long), target_lengths)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            epoch_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})

        if val_loader:
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for images, targets, target_lengths, labels in val_loader:
                    preds = model(images.to(Config.DEVICE))
                    decoded = decode_predictions(preds, Config.IDX2CHAR)
                    for p, g in zip(decoded, labels):
                        if p == g: correct += 1
                        total += 1
            val_acc = (correct / total) * 100
            print(f"Val Acc: {val_acc:.2f}%")
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model.state_dict(), "best_model.pth")


In [ ]:
if __name__ == "__main__":
    train_pipeline()

🔒 Đã cố định Seed: 42
🚀 TRAINING START | Device: cuda
[TRAIN] Scanning: ../data/train
📂 Loading splits from 'val_tracks.json' and 'test_tracks.json'...
[TRAIN] Loaded 16000 tracks.


Indexing train: 100%|██████████| 16000/16000 [00:04<00:00, 3541.72it/s]


[VAL] Scanning: ../data/train
📂 Loading splits from 'val_tracks.json' and 'test_tracks.json'...
[VAL] Loaded 2000 tracks.


Indexing val: 100%|██████████| 2000/2000 [00:00<00:00, 3739.39it/s]


[TEST] Scanning: ../data/train
📂 Loading splits from 'val_tracks.json' and 'test_tracks.json'...
[TEST] Loaded 2000 tracks.


Ep 1/50:   0%|          | 0/250 [00:00<?, ?it/s]